# CHIRPS–ET–Groundwater Analysis Script

This script processes CHIRPS precipitation data, extracts site-level precipitation using spatial buffers, and combines it with ET and USGS groundwater data to compute ET:precipitation ratios and groundwater decline trends across study sites.

In [1]:
# Import libraries
import os
import requests
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
from scipy.stats import spearmanr
import glob
from rasterio.mask import mask
from shapely.geometry import Point, box
from pyproj import Transformer

In [2]:
# Read in ET and groundwater data for all sites
et_gw_merged_all_sites = "/capstone/aridgw/outputs/4km/gwl_cult_et_4km.csv"
et_gw_merged_all_sites = pd.read_csv(et_gw_merged_all_sites)
et_gw_merged_all_sites

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,678.880
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,668.881
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,639.741
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,704.653
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,671.125
...,...,...,...,...,...,...,...,...,...,...,...
986,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,725.501
987,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,743.316
988,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,726.933
989,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,761.197


## Creating a precipitation normal raster using 2000–2020 CHIRPS data to match the temporal extent of the study dataset.

In [3]:
# Takes 23 mins to run

# Define folder containing daily CHIRPS raster files
data_dir = "/capstone/aridgw/data/chirps_daily_data/"

# Select only CHIRPS raster files from 2000–2020
files = sorted(glob.glob(data_dir + "*.tif"))
files = [f for f in files if any(str(year) in f for year in range(2000, 2021))]
print(len(files), "files found")

# Read first file to get metadata
with rasterio.open(files[0]) as src:
    meta = src.meta.copy()
    shape = src.shape

# Create array to store summed precipitation values
precip_sum = np.zeros(shape, dtype=np.float64)

# Loop through rasters and sum precipitation values
for i, f in enumerate(files):
    with rasterio.open(f) as src:
        precip_sum += src.read(1)
    if i % 100 == 0:
        print(f"Processed {i}/{len(files)}")

# Calculate mean daily precipitation raster
precip_mean = precip_sum / len(files)

# Save precipitation normal raster to a remote server
meta.update(dtype=rasterio.float32)
with rasterio.open("/capstone/aridgw/data/chirps_daily_precip_normal_2000_2020.tif", 'w', **meta) as dst:
    dst.write(precip_mean.astype(np.float32), 1)

# Save precipitation normal raster locally to `outputs` folder
#meta.update(dtype=rasterio.float32)
#with rasterio.open("../outputs/chirps_daily_precip_normal_2000_2020.tif", 'w', **meta) as dst:
#    dst.write(precip_mean.astype(np.float32), 1)

0 files found


IndexError: list index out of range

##  The mean is calculated by averaging all raster pixel values that intersect the buffered region, with each intersecting pixel contributing equally regardless of how much of its area overlaps the buffer.

## To change the buffer size, edit `buffer_km=0.5` to half the length of the buffer you want (example: for a 10 km buffer use `buffer_km=5`) in the following line:
`def extract_precip_with_buffer(lon, lat, src, buffer_km=0.5):`

## When changing buffer size, you have to also update all related file names and labels, including:
-   Output filenames (e.g., 1km → 2km, 4km, 10km)
-   Variable names that include resolution or scale (e.g., et_precip_ratio_1km)
-   Folder names if used (e.g., /outputs/1km/)

The computation code (masking, CRS transform, mean extraction) does not need to change—only the buffer parameter and naming conventions.

In [4]:
et_gw_merged_all_sites = et_gw_merged_all_sites.copy()

# Define function to extract precipitation values around each site
# Change `buffer_km=0.5` ⚠️
def extract_precip_with_buffer(lon, lat, src, buffer_km=2):

    # Transform coordinates to raster coordinate system
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    x, y = transformer.transform(lon, lat)

    # Create square buffer around site
    half_size = buffer_km / 111.32
    geom = box(
        x - half_size, y - half_size,
        x + half_size, y + half_size
    )
    
    # Mask raster to buffered site area
    out_image, out_transform = mask(
        src,
        [geom],
        crop=True,
        all_touched=True
    )

    data = out_image[0]

    # Return mean precipitation value for buffered area
    return data.mean() if data.size > 0 else np.nan

# Extract precipitation values for all sites
# Change `buffer_km=0.5` ⚠️
with rasterio.open("/capstone/aridgw/outputs/chirps_daily_precip_normal_2000_2020.tif") as src:
    precip_values = [
        extract_precip_with_buffer(lon, lat, src, buffer_km=2)
        for lon, lat in zip(
            et_gw_merged_all_sites["longitude"],
            et_gw_merged_all_sites["latitude"]
        )
    ]

# Add precipitation normals to dataframe
et_gw_merged_all_sites["chirps_precip_normal"] = precip_values
et_gw_merged_all_sites

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg,chirps_precip_normal
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,678.880,1.511063
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,668.881,1.511063
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,639.741,1.511063
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,704.653,1.511063
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,671.125,1.511063
...,...,...,...,...,...,...,...,...,...,...,...,...
986,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,725.501,0.912996
987,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,743.316,0.912996
988,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,726.933,0.912996
989,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,761.197,0.912996


In [5]:
# Create copy of dataframe for groundwater trend calculations
df = et_gw_merged_all_sites.copy()

# Define function to calculate groundwater trends
def calc_trend(group):
    # Remove rows with missing groundwater values
    g = group.dropna(subset=["year_value", "depth_to_gw_m"])
    if len(g) < 2:
        return np.nan
    
    # Calculate groundwater trend using linear regression slope (m/year)
    slope = np.polyfit(g["year_value"], g["depth_to_gw_m"], 1)[0]
    return slope

# Calculate groundwater trends for each site
gw_trends = df.groupby("site_id").apply(calc_trend).reset_index()
gw_trends.columns = ["site_id", "gw_trend_m_per_yr"]

# Calculate climate variables for each site
climate = df.groupby("site_id").agg(
    region=("region", "first"),
    mean_et=("scaled_annual_et_avg", "mean"),
    mean_precip=("chirps_precip_normal", "mean")
).reset_index()

# Multiply by 365 to get from mm/day to mm/year
climate["mean_precip"] = climate["mean_precip"] * 365

# Calculate ET to precipitation ratio
climate["et_precip_ratio"] = climate["mean_et"] / climate["mean_precip"]

# Merge groundwater trends with climate variables
site_summary = gw_trends.merge(climate, on="site_id")
site_summary

/tmp/ipykernel_1224283/1897879128.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gw_trends = df.groupby("site_id").apply(calc_trend).reset_index()


,site_id,gw_trend_m_per_yr,region,mean_et,mean_precip,et_precip_ratio
0,KSGS.371852100505801,0.616701,Southern Kansas,702.412190,551.538147,1.273551
1,KSGS.372043101363101,0.213301,Southern Kansas,560.216095,515.048828,1.087695
2,KSGS.372539100142504,1.047972,Southern Kansas,830.360381,605.336914,1.371733
3,KSGS.373331098033301,0.063359,Southern Kansas,807.138000,935.923584,0.862397
4,KSGS.373607100565301,0.595615,Southern Kansas,671.167714,523.986206,1.280888
5,KSGS.374111099070401,0.176172,Southern Kansas,801.662905,737.624390,1.086817
6,KSGS.374125100344101,1.690761,Southern Kansas,705.153714,580.897888,1.213903
7,KSGS.374747100552101,1.912739,Southern Kansas,848.274333,526.280457,1.611829
8,KSGS.375145100485701,1.217462,Southern Kansas,795.615952,537.578613,1.479999
9,KSGS.375454101075401,1.541198,Southern Kansas,930.159476,529.743164,1.755869


In [6]:
# Save site_summary as a csv to outputs folder in remote server
# Change file paths ⚠️
site_summary.to_csv("/capstone/aridgw/outputs/4km/site_summary_4km.csv", index=False)

# Save site_summary as a csv to outputs folder locally
# Change file paths ⚠️
site_summary.to_csv("../outputs/site_summary_4km.csv", index=False)

In [7]:
# Merge summary variables back into full dataset
et_gw_merged_all_sites = et_gw_merged_all_sites.merge(
    site_summary[['site_id', 'mean_et', 'mean_precip', 'et_precip_ratio']],
    on = 'site_id',
    how = 'left'
)

# Remove temporary precipitation normal column
et_gw_merged_all_sites = et_gw_merged_all_sites.drop(columns="chirps_precip_normal")
et_gw_merged_all_sites

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg,mean_et,mean_precip,et_precip_ratio
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,678.880,702.412190,551.538147,1.273551
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,668.881,702.412190,551.538147,1.273551
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,639.741,702.412190,551.538147,1.273551
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,704.653,702.412190,551.538147,1.273551
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,671.125,702.412190,551.538147,1.273551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,725.501,716.120583,333.243561,2.148940
987,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,743.316,716.120583,333.243561,2.148940
988,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,726.933,716.120583,333.243561,2.148940
989,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,761.197,716.120583,333.243561,2.148940


In [8]:
# Save et_gw_merged_all_sites as a csv to outputs folder in remote server
# Change file paths ⚠️
et_gw_merged_all_sites.to_csv(
    "/capstone/aridgw/outputs/4km/et_precipt_ratio_4km.csv",
    index = False
)

# Save et_gw_merged_all_sites as a csv to outputs folder locally
# Change file paths ⚠️
et_gw_merged_all_sites.to_csv(
    "../outputs/et_precipt_ratio_4km.csv",
    index = False
)

In [9]:
# Read in final ET to precipitation ratio dataset
# Change file paths ⚠️
et_gw_merged_all_sites = "/capstone/aridgw/outputs/4km/et_precipt_ratio_4km.csv"
et_gw_merged_all_sites = pd.read_csv(et_gw_merged_all_sites)
et_gw_merged_all_sites

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg,mean_et,mean_precip,et_precip_ratio
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,678.880,702.412190,551.53815,1.273551
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,668.881,702.412190,551.53815,1.273551
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,639.741,702.412190,551.53815,1.273551
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,704.653,702.412190,551.53815,1.273551
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,671.125,702.412190,551.53815,1.273551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,725.501,716.120583,333.24356,2.148940
987,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,743.316,716.120583,333.24356,2.148940
988,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,726.933,716.120583,333.24356,2.148940
989,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,Southwest US,4,2.0,761.197,716.120583,333.24356,2.148940
